# Creating the Bulls Draft Data

---

#### Importing Raw Draft Data

Raw data found on https://www.basketball-reference.com/teams/CHI/draft.html

Read in html table using pandas built in read html function. Also generated PLayer URLs from the player name, and then search through each player page for their rookie seaosn stats. This will take a longer time because a manual timer was added in, as to not get jailed from the site for botting. 

In [19]:
import pandas as pd
from pandas.errors import EmptyDataError
from pathlib import Path
import time

url = 'https://www.basketball-reference.com/teams/CHI/draft.html'
df = pd.read_html(url, attrs={'id': 'draft'})[0]
if isinstance(df.columns, pd.MultiIndex):
    new_cols = []
    seen = {}
    for top, sub in df.columns:
        if sub in seen:
            seen[sub] += 1
            col_name = f"{sub}.{seen[sub]-1}"
        else:
            seen[sub] = 1
            col_name = sub
        new_cols.append(col_name)
    df.columns = new_cols

df = df[df['Year'] != 'Year'].copy()
df = df[df['Year'].astype(int) >= 2000]
df = df.reset_index(drop=True)

df.head()

,Year,Lg,Rd,Pk,Player,College,G,MP,PTS,TRB,...,3P%,FT%,MP.1,PTS.1,TRB.1,AST.1,WS,WS/48,BPM,VORP
0,2025,NBA,1,12,Noa Essengue,NaN,2,6,0,0,...,.000,NaN,3.0,0.0,0.0,0.0,-0.1,-0.417,-23.7,0.0
1,2025,NBA,2,45,Rocco Zikarsky (↳LAL ↳MIN),NaN,5,36,14,14,...,NaN,1.000,7.2,2.8,2.8,0.4,0.2,.237,2.4,0.0
2,2024,NBA,1,11,Matas Buzelis,NaN,157,3759,1940,726,...,.353,.795,23.9,12.4,4.6,1.5,4.4,.057,-1.6,0.4
3,2023,NBA,2,59,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022,NBA,1,18,Dalen Terry,Arizona,218,2433,767,360,...,.314,.638,11.2,3.5,1.7,1.2,3.1,.061,-2.6,-0.3


This is the raw data from basketball-reference.

There are a couple of problems with the data:
- Includes draft picks that were traded to other teams
- Includes present day stats, we need players' rookie year stats
- Some years have no draft picks, need to disregard

#### Drop Null Picks

Some seasons resulted in zero draft picks, but these are represented by empty rows

In [20]:
print(f'Full dataset shape: {df.shape[0]} rows')

draft_df = df.dropna(subset=['Player']).reset_index(drop=True)
print(f'Dataset after removing rows with missing Player names: {draft_df.shape[0]} rows')

Full dataset shape: 55 rows
Dataset after removing rows with missing Player names: 54 rows


#### Getting Rid of Traded Players


In [21]:
print(f'Full dataset shape: {draft_df.shape[0]} rows')

draft_df = draft_df[~draft_df['Player'].str.contains('↳')].reset_index(drop=True)
print(f'Dataset after removing traded players: {draft_df.shape[0]} rows')

Full dataset shape: 54 rows
Dataset after removing traded players: 39 rows


#### Creating Rookie Year Data Folders

A place to put all of the rookie year stats is needed to store all of the data. A folder was made manually to store csvs, but each csv file was made programatically based on what players are available. 

Some of the players' stats have different columns based on how much playtime they had, so the stats couldn;t be grouped into the same csv file right away. 

In [22]:
# # define folder to save files in
# output_dir = Path('../data/rookie_stats_data')
# output_dir.mkdir(parents=True, exist_ok=True)

# # gather player names
# player_names = (
#     draft_df['Player']
#     .dropna()
#     .astype(str)
#     .str.strip()
#     .drop_duplicates()
# )

# created_files = []
# existing_files = []

# # create a CSV file for each player
# for player_name in player_names:
#     # file name is in the structure {first name}_{last_name}.csv
#     filename = f"{player_name.replace(' ', '_')}.csv"
#     file_path = output_dir / filename

#     if file_path.exists():
#         existing_files.append(filename)
#         continue

#     file_path.write_text('')
#     created_files.append(filename)

# print(f"Created {len(created_files)} CSV files in {output_dir}")
# print(f"Skipped {len(existing_files)} existing files")


#### Cleaning the Rookie Stats Files

Some of the pages for the players' rookie years were empty or missing certain statistics. Those columns needed to be added and set to be null so that the rookie stats could be merged correctly.

In [23]:
# create the rookie_stats_data directory (starting empty)
output_dir = Path('../data/raw/rookie_stats_data')
output_dir.mkdir(parents=True, exist_ok=True)

# fetch data for each player
for index, row in draft_df.iterrows():
    player_name = row['Player']
    # parse first and last name
    parts = player_name.split()
    if len(parts) < 2:
        continue
    first_name = parts[0]
    last_name = parts[-1]
    # clean names for URL (remove non-alphanumeric, lowercase)
    clean_last = ''.join(c.lower() for c in last_name if c.isalnum())
    clean_first = ''.join(c.lower() for c in first_name if c.isalnum())
    if not clean_last or not clean_first:
        continue
    first_letter = clean_last[0]
    last_5 = clean_last[:5]
    first_2 = clean_first[:2]
    player_code = f"{last_5}{first_2}01"
    url = f"https://www.basketball-reference.com/players/{first_letter}/{player_code}.html"
    print(f"Fetching data for {player_name} from {url}")
    try:
        tables = pd.read_html(url)
        advanced_df = None
        for table in tables:
            if "BPM" in table.columns and "VORP" in table.columns:
                advanced_df = table
                break
        if advanced_df is None:
            raise ValueError("Advanced table not found.")
        advanced_df = advanced_df[advanced_df["Season"] != "Season"]
        # rookie year is the first row
        rookie_data = advanced_df.iloc[0:1]
        # save to CSV with the same naming convention
        filename = f"{player_name.replace(' ', '_')}.csv"
        file_path = output_dir / filename
        rookie_data.to_csv(file_path, index=False)
        print(f"Saved data for {player_name} to {file_path}")
    except Exception as e:
        print(f"Failed to fetch data for {player_name}: {e}")
    time.sleep(6)


Fetching data for Noa Essengue from https://www.basketball-reference.com/players/e/essenno01.html
Saved data for Noa Essengue to ..\data\raw\rookie_stats_data\Noa_Essengue.csv
Fetching data for Matas Buzelis from https://www.basketball-reference.com/players/b/buzelma01.html
Saved data for Matas Buzelis to ..\data\raw\rookie_stats_data\Matas_Buzelis.csv
Fetching data for Dalen Terry from https://www.basketball-reference.com/players/t/terryda01.html
Saved data for Dalen Terry to ..\data\raw\rookie_stats_data\Dalen_Terry.csv
Fetching data for Ayo Dosunmu from https://www.basketball-reference.com/players/d/dosunay01.html
Saved data for Ayo Dosunmu to ..\data\raw\rookie_stats_data\Ayo_Dosunmu.csv
Fetching data for Patrick Williams from https://www.basketball-reference.com/players/w/willipa01.html
Saved data for Patrick Williams to ..\data\raw\rookie_stats_data\Patrick_Williams.csv
Fetching data for Marko Simonovic from https://www.basketball-reference.com/players/s/simonma01.html
Saved data

In [24]:
# define folder to read files from
data_dir = Path('../data/raw/rookie_stats_data')
csv_files = sorted(data_dir.glob('*.csv'))

file_data = {}
all_columns = []

# read each CSV file and collect column information
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        file_data[csv_file] = df

        for column in df.columns:
            if column not in all_columns:
                all_columns.append(column)
    except EmptyDataError:
        file_data[csv_file] = None

# unify columns across all files
if not all_columns:
    print('No column information was found in the CSV files.')
else:
    updated_files = []
    empty_files = []

    # fill missing columns with NaN and save updated files
    for csv_file, df in file_data.items():
        if df is None:
            fixed_df = pd.DataFrame([{column: pd.NA for column in all_columns}])
            empty_files.append(csv_file.name)
            missing_columns = all_columns.copy()
        else:
            missing_columns = [column for column in all_columns if column not in df.columns]
            fixed_df = df.copy()

            for column in missing_columns:
                fixed_df[column] = pd.NA

            fixed_df = fixed_df[all_columns]

        fixed_df.to_csv(csv_file, index=False)

        if missing_columns:
            updated_files.append((csv_file.name, missing_columns))

    print(f'Checked {len(csv_files)} files.')
    print(f'Unified column set has {len(all_columns)} columns.\n')
    

    # report results
    if empty_files:
        print('Empty files filled with NaN values:\n')
        for file_name in empty_files:
            print(f'  {file_name}')
        

    if updated_files:
        print('Files updated with missing columns:\n')
        for file_name, missing_columns in updated_files:
            print(f'  {file_name}: {missing_columns}')
    else:
        print('All files already had the same columns.')


Checked 39 files.
Unified column set has 51 columns.

Files updated with missing columns:

  A.J._Guyton.csv: ['FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']
  Aaron_Gray.csv: ['FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']
  Ayo_Dosunmu.csv: ['FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']
  Ben_Gordon.csv: ['FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']
  Bobby_Portis.csv: ['FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']
  Cameron_Bairstow.csv: ['PER', 'TS%', '3PAr', 'FTr', 'ORB%'

#### Getting Rookie Year Data

Stats are currently listed as most recent season, and this project requires the rookie year stats. This was not possible programatically, so all stats were pasted in manually.

The old stats from recent seasons need to be dropped, and the rookie year season needs to be added in for each player.

In [25]:
print(f'Current dataset shape: {draft_df.shape[1]} columns \n')

# removing columns that show present stats
draft_df = draft_df.drop(columns=['G','MP','PTS','TRB','AST','FG%','3P%','FT%','MP.1','PTS.1','TRB.1','AST.1','WS','WS/48','BPM','VORP'])
print(f'Removed columns. The columns kept are {draft_df.columns.tolist()} which is {draft_df.shape[1]} columns\n')

# columns to be added to draft_df from rookie year stats
columns_list = ['Age', 'Pos', 'G', 'GS', 'MP', 'FG', 'FGA',
       'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA',
       'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS',
       'Awards']

print(f'Adding in {len(columns_list)} columns to the dataset, imported from rookie year stats\n')

# add missing columns to draft_df with NaN values
for col in columns_list:
    if col not in draft_df.columns:
        draft_df[col] = float('nan')

# read each player's CSV file and update the corresponding row in draft_df
for csv_file in sorted(data_dir.glob('*.csv')):
    player_name = csv_file.stem.replace('_', ' ')
    # create a boolean mask to identify the row in draft_df corresponding to the current player
    row_mask = draft_df['Player'].eq(player_name)

    if not row_mask.any():
        continue

    # read the player's CSV file into a DataFrame
    player_df = pd.read_csv(csv_file)
    if player_df.empty:
        continue

    # identify which columns from columns_list are present in the player's DataFrame
    available_cols = [col for col in columns_list if col in player_df.columns]
    non_empty_rows = player_df[available_cols].dropna(how='all')

    if non_empty_rows.empty:
        continue

    # take the first non-empty row of the player's DataFrame and update the corresponding row in draft_df
    player_row = non_empty_rows.iloc[0]
    draft_df.loc[row_mask, available_cols] = player_row[available_cols].values

draft_df


Current dataset shape: 22 columns 

Removed columns. The columns kept are ['Year', 'Lg', 'Rd', 'Pk', 'Player', 'College'] which is 6 columns

Adding in 28 columns to the dataset, imported from rookie year stats



C:\Users\12248\AppData\Local\Temp\ipykernel_22044\1426846571.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'PG' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  draft_df.loc[row_mask, available_cols] = player_row[available_cols].values
C:\Users\12248\AppData\Local\Temp\ipykernel_22044\1426846571.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ROY-2,6MOY-1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  draft_df.loc[row_mask, available_cols] = player_row[available_cols].values
C:\Users\12248\AppData\Local\Temp\ipykernel_22044\1426846571.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Did not play - other pro league' has dtype incompatible wit

,Year,Lg,Rd,Pk,Player,College,Age,Pos,G,GS,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,2025,NBA,1,12,Noa Essengue,NaN,19,PF,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024,NBA,1,11,Matas Buzelis,NaN,20,SF,80,31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ROY-7
2,2022,NBA,1,18,Dalen Terry,Arizona,20.0,SG,38.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021,NBA,2,38,Ayo Dosunmu,Illinois,22.0,SG,77.0,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020,NBA,1,4,Patrick Williams,Florida State,19,PF,71,71,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2020,NBA,2,44,Marko Simonovic,NaN,Did not play - other pro league,Did not play - other pro league,Did not play - other pro league,Did not play - other pro league,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Did not play - other pro league
6,2019,NBA,1,7,Coby White,UNC,19.0,PG,65.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ROY-5
7,2019,NBA,2,38,Daniel Gafford,Arkansas,21.0,C,43.0,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2018,NBA,1,7,Wendell Carter,Duke,19,C,44.0,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2018,NBA,1,22,Chandler Hutchison,Boise State,22.0,SF,44.0,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Saving the Data

The dataset has the rookie stats of all the players, now the dataset needs to be saved. It's next step is to be cleaned and prepared for analysis

In [26]:
draft_df.to_csv('../data/processed/draft_data_cleaned.csv', index=False)